| Тип     | Описание             | Применение                                   |
| ------- | -------------------- | -------------------------------------------- |
| tag     | Теги/категории       | Фильтрация Tag("user") == "john", сортировка |
| text    | Полнотекстовый поиск | Поиск по словам, стемминг                    |
| vector  | Векторные embeddings | КНН-поиск, HNSW                              |
| numeric | Числовые значения    | Диапазонные запросы > 700                    |
| geo     | Геолокация           | Поиск по расстоянию                          |
| json    | JSON объекты         | Структурированные данные                     |

In [94]:
from redisvl.schema import IndexSchema

schema = IndexSchema.from_yaml("schemas/schema.yaml")

In [95]:
from redis import Redis
from redisvl.index import SearchIndex

# Define the index
index = SearchIndex(schema, redis_url="redis://:password@localhost:6383")

# Create the index in Redis
index.create()

In [ ]:
data = {"doc_id": "v1", "user": "john", "credit_score": "high", "embedding": [0.23, 0.49, -0.18, 0.95]}

# load list of dictionaries, specify the "id" field
index.load([data], id_field="doc_id")

# fetch by "id"
john = index.fetch("v1")
john

{'doc_id': 'v1',
 'user': 'john',
 'credit_score': 'high',
 'embedding': [0.23, 0.49, -0.18, 0.95]}

In [97]:
docs = [
    {
        "doc_id": "v100",
        "user": "user1",
        "credit_score": "850",
        "job_title": "Senior Engineer", 
        "embedding": [0.1, 0.2, 0.3, 0.4]
    },
    {
        "doc_id": "v2",
        "user": "john", 
        "credit_score": "720",
        "job_title": "Data Scientist",
        "embedding": [0.16, -0.34, 0.98, 0.23]
    }
]

index.load(docs, id_field="doc_id")

['user:v100', 'user:v2']

In [98]:
from redisvl.query import VectorQuery

query = VectorQuery(
  vector=[0.16, -0.34, 0.98, 0.23],
  vector_field_name="embedding",
  num_results=3,
  # Optional: tune search performance with runtime parameters
  # ef_runtime=100  # HNSW: higher for better recall
)
# run the vector search query against the embedding field
results = index.query(query)

results

[{'id': 'user:v2', 'vector_distance': '0'},
 {'id': 'user:v100', 'vector_distance': '0.432469904423'},
 {'id': 'user:v1', 'vector_distance': '1.07365822792'}]

In [112]:
from redisvl.query.filter import Tag

# define a tag match filter
tag_filter = Tag("user") == "john"

# update query definition
query.set_filter(tag_filter)

# execute query
results = index.query(query)

for result in results:
    doc_id = result['id'].split(':')[1]  # 'user:v2'
    full_doc = index.fetch(doc_id)  # Загружает ВСЕ поля
    
    print("ID:", doc_id)
    print("Полные данные:", full_doc)
    print("-" * 50)

ID: v2
Полные данные: {'doc_id': 'v2', 'user': 'john', 'credit_score': '720', 'job_title': 'Data Scientist', 'embedding': [0.16, -0.34, 0.98, 0.23]}
--------------------------------------------------
ID: v1
Полные данные: {'doc_id': 'v1', 'user': 'john', 'credit_score': 'high', 'embedding': [0.23, 0.49, -0.18, 0.95]}
--------------------------------------------------


In [ ]:
# index.clear()

3

In [ ]:
results = index.query(query)

results

[]

# Семантическое кэширование

In [128]:
from redisvl.extensions.cache.llm import SemanticCache

# init cache with TTL and semantic distance threshold
llmcache = SemanticCache(
    name="llmcache",
    ttl=360,
    redis_url="redis://:password@localhost:6383",
    distance_threshold=0.5  # Redis COSINE distance [0-2], lower is stricter
)

# store user queries and LLM responses in the semantic cache
llmcache.store(
    prompt="What is France's capital city?",
    response="Paris"
)

llmcache.store(
    prompt="What is this?",
    response="Repeat, please"
)

# quickly check the cache with a slightly different prompt (before invoking an LLM)
response = llmcache.check(prompt="What is this")
print(response[0]["response"])

Repeat, please


# Кэширование эмбеддингов

In [130]:
from redisvl.extensions.cache.embeddings import EmbeddingsCache
from redisvl.utils.vectorize import HFTextVectorizer

# Initialize embedding cache
embed_cache = EmbeddingsCache(
    name="embed_cache",
    redis_url="redis://:password@localhost:6383",
    ttl=3600  # 1 hour TTL
)

# Initialize vectorizer with cache
vectorizer = HFTextVectorizer(
    model="sentence-transformers/all-MiniLM-L6-v2",
    cache=embed_cache
)

# First call computes and caches the embedding
embedding = vectorizer.embed("What is machine learning?")

# Subsequent calls retrieve from cache (much faster!)
cached_embedding = vectorizer.embed("What is machine learning?")

# LLM Память 

In [131]:
from redisvl.extensions.message_history import SemanticMessageHistory

history = SemanticMessageHistory(
    name="my-session",
    redis_url="redis://:password@localhost:6383",
    distance_threshold=0.7
)

# Supports roles: system, user, llm, tool
# Optional metadata field for additional context
history.add_messages([
    {"role": "user", "content": "hello, how are you?"},
    {"role": "llm", "content": "I'm doing fine, thanks."},
    {"role": "user", "content": "what is the weather going to be today?"},
    {"role": "llm", "content": "I don't know", "metadata": {"model": "gpt-4"}}
])

c:\main\data_engineering\data-engineering_practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tumbi\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

In [132]:
history.get_recent(top_k=1)

[{'role': 'llm', 'content': "I don't know", 'metadata': {'model': 'gpt-4'}}]

In [133]:
history.get_relevant("weather", top_k=1)

[{'role': 'user', 'content': 'what is the weather going to be today?'}]

In [134]:
# Get only user messages
history.get_recent(role="user")
# Or multiple roles
history.get_recent(role=["user", "system"])

[{'role': 'user', 'content': 'hello, how are you?'},
 {'role': 'user', 'content': 'what is the weather going to be today?'}]

# Семантическая маршрутизация

In [139]:
from redisvl.extensions.router import Route, SemanticRouter

routes = [
    Route(
        name="greeting",
        references=["hello", "hi"],
        metadata={"type": "greeting"},
        distance_threshold=0.3,
    ),
    Route(
        name="farewell",
        references=["bye", "goodbye"],
        metadata={"type": "farewell"},
        distance_threshold=0.3,
    ),
]

# build semantic router from routes
router = SemanticRouter(
    name="topic-router",
    routes=routes,
    redis_url="redis://:password@localhost:6383",
)


router("Hi, good morning")

RouteMatch(name='greeting', distance=0.273891299963)

## Задание 1: Создание векторного индекса

Цель: Создать индекс для хранения векторных представлений текстов.
Шаги:

Подключитесь к Redis.
Создайте векторный индекс с параметрами:

Размерность векторов: 384.
Метрика расстояния: косинусное.

Добавьте несколько векторов в индекс.


In [140]:
from redis import Redis
from redisvl.schema import IndexSchema
from redisvl.index import SearchIndex

schema = IndexSchema.from_dict({
    "index": {
        "name": "docs",
        "prefix": "doc",
        "storage_type": "json"
    },
    "fields": [
        {"name": "doc_idx", "type": "tag"},
        {"name": "category", "type": "tag"},
        {
            "name": "content",
            "type": "text",
            "attrs": {
                "sortable": True,
                "no_index": False,
                "unf": False
            }
        },
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "hnsw",
                "dims": 384,
                "distance_metric": "cosine",
                "datatype": "float32"
            }
        }
    ]
})

# Define the index
index = SearchIndex(schema, redis_url="redis://:password@localhost:6383")

# Create the index in Redis
index.create()

In [141]:
from faker import Faker
import numpy as np
from sentence_transformers import SentenceTransformer

fake = Faker('ru_RU')  # Русские данные
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # 384 dims

# Категории для реалистичности
categories = ['технологии', 'финансы', 'медицина', 'образование', 'маркетинг']

def generate_synthetic_docs(n_docs=1000):
    docs = []
    for i in range(n_docs):
        doc = {
            "doc_idx": f"doc_{i:04d}",
            "category": np.random.choice(categories),
            "content": fake.text(max_nb_chars=500),
        }
        
        # Генерация embedding из текста
        embedding = model.encode(doc["content"]).tolist()
        doc["embedding"] = embedding
        
        docs.append(doc)
        if i % 100 == 0:
            print(f"Сгенерировано: {i}/{n_docs}")
    
    return docs

# Генерируем и загружаем
print("Генерация синтетических данных...")
synthetic_docs = generate_synthetic_docs(500)

print("Загрузка в Redis...")
index.load(synthetic_docs, id_field="doc_idx")

print(f"✅ Загружено {len(synthetic_docs)} документов!")


c:\main\data_engineering\data-engineering_practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tumbi\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xe

Генерация синтетических данных...
Сгенерировано: 0/500
Сгенерировано: 100/500
Сгенерировано: 200/500
Сгенерировано: 300/500
Сгенерировано: 400/500
Загрузка в Redis...
✅ Загружено 500 документов!


In [150]:
# Поиск по категории
from redisvl.query import VectorQuery
from redisvl.query.filter import Tag

query_text = "искусственный интеллект"
query_embedding = model.encode(query_text).tolist()

query = VectorQuery(
    vector=query_embedding,
    vector_field_name="embedding",
    num_results=5
)

# Только категория 'технологии'
tag_filter = Tag("category") == "технологии"
query.set_filter(tag_filter)

results = index.query(query)
for r in results:
    doc_id = r['id'].split(':')[1]
    doc = index.fetch(doc_id)
    print(f"ID: {r['id']}, Distance: {float(r['vector_distance']):.3f}")
    print(f"  Категория: {doc['category']}")
    print(f"  Текст: {doc['content'][:100]}...")
    print()


ID: doc:doc_0101, Distance: 0.595
  Категория: технологии
  Текст: Смертельный войти функция космос торговля идея число. Близко висеть палец космос поколение. Болото ц...

ID: doc:doc_0265, Distance: 0.607
  Категория: технологии
  Текст: Необычный иной трясти пятеро скрытый аллея. Решетка вздрагивать поколение рис пропасть. Команда набо...

ID: doc:doc_0069, Distance: 0.619
  Категория: технологии
  Текст: Волк оставить другой. Лететь основание свежий вообще рай дорогой. Настать левый металл.
Устройство т...

ID: doc:doc_0360, Distance: 0.629
  Категория: технологии
  Текст: Ложиться да достоинство терапия наслаждение выбирать изображать. Освобождение слишком девка полюбить...

ID: doc:doc_0312, Distance: 0.632
  Категория: технологии
  Текст: Стакан отдел засунуть палка карман актриса. Тута указанный вытаскивать упор отметить.
Палка дрогнуть...



## Задание 2: Поиск ближайших соседей

Цель: Найти 3 самых похожих вектора для заданного запроса.
Шаги:

Сгенерируйте вектор для запроса (например, с помощью sentence-transformers).
Выполните поиск в индексе.
Выведите результаты.


## Задание 3: Обновление и удаление векторов

Цель: Обновить вектор для существующего тега и удалить вектор по тегу.
Шаги:

Обновите вектор для тега text1.
Удалите вектор с тегом text2.


## Задание 4: Интеграция с RAG

Цель: Использовать RedisVL для поиска релевантных документов в системе RAG.
Шаги:

Загрузите датасет документов.
Сгенерируйте векторы для документов и сохраните их в RedisVL.
Для запроса найдите топ-3 релевантных документа.
Передайте найденные документы в LLM для генерации ответа.


## 4. Дополнительные задания

Оптимизация индекса: Попробуйте использовать HNSW вместо FLAT для ускорения поиска.
Батчинг: Реализуйте пакетную загрузку векторов в RedisVL.
Мониторинг производительности: Измерьте время поиска для разных размеров индекса.
